# CaptureTrust in Google Colab

Inspect an uploaded image for visual quality and get an explainable report.

Images are processed locally in this Colab runtime and are **not** uploaded anywhere by the library.

In [ ]:
!pip install capturetrust

In [ ]:
from google.colab import files
from capturetrust import inspect_image

uploaded = files.upload()
filename = next(iter(uploaded))

report = inspect_image(filename)

print(report.to_json())

In [ ]:
# Human-readable summary
print(report.summary())

for issue, tip in zip(report.issues, report.recommendations):
    print(f'- {issue} -> {tip}')

## Check many photos at once (plain-language summary)

Upload several photos and see, in plain English, how each one was judged. This is the easiest way to sanity-check the tool on your own images.

In [ ]:
import os
from google.colab import files
from capturetrust import inspect_image

os.makedirs('my_photos', exist_ok=True)
print('Select several photos to upload...')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join('my_photos', name), 'wb') as f:
        f.write(data)

paths = sorted('my_photos/' + n for n in uploaded)
passed = 0
print(f"\n{'Photo':<28}{'Score':>6}  Verdict       Notes")
print('-' * 74)
for p in paths:
    r = inspect_image(p)
    verdict = 'looks good' if r.is_acceptable else 'needs review'
    notes = '; '.join(r.issues) if r.issues else 'no issues found'
    passed += int(r.is_acceptable)
    print(f"{p.split('/')[-1]:<28}{r.score:>6}  {verdict:<13} {notes}")
print('-' * 74)
print(f"\n{passed} of {len(paths)} looked good; {len(paths)-passed} need review.")
print('Compare each verdict to what your own eyes say - that tells you if it is calibrated well for your photos.')

In [ ]:
# Customise thresholds
from capturetrust import CaptureTrustConfig, inspect_image

config = CaptureTrustConfig(min_width=1024, min_height=768, acceptable_score=75)
report = inspect_image(filename, config=config)
print(report.score, report.is_acceptable)